# VoiceDiary AI — Live Cloud GPU Platform
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

> **Quick Start:** Click **Runtime → Run all** (`Ctrl+F9`) · Universal Silero Neural Noise Cancellation & VAD pre-gating across all models!

In [ ]:
!pip install -q --no-cache-dir faster-whisper speechbrain gradio soundfile torchaudio scipy


In [ ]:

import os, time, json, re, urllib.request, base64, gc
import numpy as np
import soundfile as sf
import gradio as gr
import torch
import torchaudio
from scipy.signal import lfilter
from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

# ─── Hardware Acceleration ───
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (AVX2)"
device_type = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = "float16" if device_type == "cuda" else "int8"
print(f"⚡ Hardware: {gpu_name} | Compute: {compute_dtype}")

# ─── Model Hub Cache ───
_model_cache = {}
def get_model(name):
    if name not in _model_cache:
        print(f"Loading Whisper model: {name}…")
        os.makedirs("/content/models/whisper", exist_ok=True)
        _model_cache[name] = WhisperModel(name, device=device_type, compute_type=compute_dtype,
            num_workers=2, download_root="/content/models/whisper")
    return _model_cache[name]

print("Pre-warming Large-v3-Turbo default model…")
get_model("large-v3-turbo")

print("Pre-warming SpeechBrain ECAPA-TDNN neural diarizer…")
os.makedirs("/content/models/ecapa", exist_ok=True)
embedder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/models/ecapa", run_opts={"device": device_type})

print("Pre-warming Universal Silero Neural VAD Noise Cancellation Model…")
silero_vad_model, silero_utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    force_reload=False,
    trust_repo=True
)
silero_vad_model.to(device_type)
silero_vad_model.eval()
print("✓ VoiceDiary AI Engine (Universal Noise Cancellation + Whisper + ECAPA) ready.")

# ─── 1. Universal Noise Cancellation & Acoustic Filtering Engine ───
# Decoupled from transcription: runs unconditionally for ALL Whisper models & inputs
w = np.tan(np.pi * 80.0 / 16000.0)
w2 = w * w
sqrt2 = np.sqrt(2.0)
norm = 1.0 + sqrt2 * w + w2
HP_B = np.array([1.0 / norm, -2.0 / norm, 1.0 / norm], dtype=np.float32)
HP_A = np.array([1.0, 2.0 * (w2 - 1.0) / norm, (1.0 - sqrt2 * w + w2) / norm], dtype=np.float32)

class UniversalAudioCleaner:
    """Universal acoustic filtering, noise floor estimation, and neural VAD gating."""
    def __init__(self, vad_model, device):
        self.vad_model = vad_model
        self.device = device
        self.ambient_rms = 0.002

    def filter_noise(self, chunk: np.ndarray) -> np.ndarray:
        """80 Hz low-cut filter removes desk vibrations, mic pops, and fan/AC rumble."""
        try:
            return lfilter(HP_B, HP_A, chunk).astype(np.float32)
        except Exception:
            return chunk

    def is_speech_active(self, chunk: np.ndarray, threshold: float = 0.40) -> bool:
        """Dynamic noise gate + Silero neural VAD evaluation."""
        if chunk is None or len(chunk) == 0:
            return False
        
        filtered = self.filter_noise(chunk)
        rms = float(np.sqrt(np.mean(filtered ** 2)))

        # Dynamic Ambient Noise Floor Gate (rejects steady fan / room hiss)
        if rms < (self.ambient_rms * 1.5) and rms < 0.0045:
            self.ambient_rms = 0.95 * self.ambient_rms + 0.05 * rms
            return False

        if len(filtered) < 512:
            return False

        try:
            tensor = torch.from_numpy(filtered).float().to(self.device)
            window_size = 512
            probs = []
            with torch.no_grad():
                for i in range(0, len(tensor) - window_size + 1, window_size):
                    frame = tensor[i : i + window_size]
                    prob = self.vad_model(frame, 16000).item()
                    probs.append(prob)
            conf = max(probs) if probs else 0.0
            if conf < threshold:
                self.ambient_rms = 0.95 * self.ambient_rms + 0.05 * rms
                return False
            return True
        except Exception:
            return rms >= 0.005

audio_cleaner = UniversalAudioCleaner(silero_vad_model, device_type)

# ─── 2. Desktop Urdu Normalizer ───
CHAR_MAPPINGS = {
    '\u0643': '\u06a9', '\u064a': '\u06cc', '\u0649': '\u06cc', '\u06c2': '\u06c1',
    '\u0647': '\u06c1', '\u06c3': '\u06c1', '\u0629': '\u06c1', '\u0624': '\u0648',
}
HAMZA_CORRECTIONS = {
    'آو': 'آؤ', 'جاو': 'جاؤ', 'کھاو': 'کھاؤ', 'پیو': 'پیئو', 'سناو': 'سناؤ', 'بتاو': 'بتاؤ',
    'دکھاو': 'دکھاؤ', 'گاو': 'گاؤ', 'لاو': 'لاؤ', 'آۓ': 'آئے', 'گۓ': 'گئے', 'ہوۓ': 'ہوئے',
    'گاۓ': 'گائے', 'بجاۓ': 'بجائے', 'جاۓ': 'جائے', 'پاۓ': 'پائے', 'کھاۓ': 'کھائے',
    'سناۓ': 'سنائے', 'بتاۓ': 'بتائے', 'دکھاۓ': 'دکھائے', 'چاہیۓ': 'چاہئے', 'کیجئے': 'کیجیئے',
    'دیجئے': 'دیجیئے', 'لیجئے': 'لیجیئے'
}
ORTHOGRAPHIC_CORRECTIONS = {
    'توتہ': 'توتا', 'گانہ': 'گانا', 'طوطا': 'توتا', 'بلکل': 'بالکل',
    'انشااللہ': 'ان شاء اللہ', 'ماشااللہ': 'ما شاء اللہ', 'صحیح': 'صحیح'
}

class UrduNormalizer:
    @staticmethod
    def normalize(text: str) -> str:
        if not text: return ""
        text = re.sub(r'[\u200b-\u200f\ufeff]', '', text)
        chars = [CHAR_MAPPINGS.get(c, c) for c in text]
        text = "".join(chars)
        for wrong, right in HAMZA_CORRECTIONS.items():
            text = re.sub(r'\b' + re.escape(wrong) + r'\b', right, text)
        for wrong, right in ORTHOGRAPHIC_CORRECTIONS.items():
            text = re.sub(r'\b' + re.escape(wrong) + r'\b', right, text)
        text = re.sub(r'\s+([۔،؟!])', r'\1', text)
        text = re.sub(r'([.?!,])\1+', r'\1', text)
        return re.sub(r'\s+', ' ', text).strip()

# ─── 3. Desktop English Post-Processor ───
ENGLISH_CONFUSION_PAIRS = [
    (r'\bavailability\s+to\s+convey\b', 'the ability to convey'),
    (r'\bavailability\s+to\b', 'ability to'),
    (r'\bfocus\s+and\s+availability\b', 'focus, and the ability'),
    (r'\bfocus\s+and\s+the\s+availability\b', 'focus, and the ability'),
    (r'\broles\s+and\s+the\s+world\b', 'role in the world'),
    (r'\brole\s+and\s+the\s+world\b', 'role in the world'),
    (r'\btheir\s+roles\s+and\s+the\s+world\b', 'their role in the world'),
    (r'\bentertains\s+inform\b', 'entertain, inform'),
    (r'\bentertains,\s*inform\b', 'entertain, inform'),
    (r'\bcontent\s+that\s+entertains\s+and\s+inform\b', 'content that entertains, informs'),
]

def post_process_english(text: str) -> str:
    if not text: return ""
    processed = text
    for pattern, replacement in ENGLISH_CONFUSION_PAIRS:
        processed = re.sub(pattern, replacement, processed, flags=re.IGNORECASE)
    processed = re.sub(r'\s+([,.:;?!])', r'\1', processed)
    processed = re.sub(r'([.?!,])\1+', r'\1', processed)
    processed = re.sub(r'\s+', ' ', processed).strip()
    if processed:
        processed = processed[0].upper() + processed[1:]
    return processed

# ─── 4. Desktop Romanizer ───
URDU_WORD_DICT = {
    'میں': 'mein', 'ہم': 'hum', 'تم': 'tum', 'آپ': 'aap', 'تو': 'tu', 'یہ': 'yeh', 'وہ': 'woh',
    'اس': 'is', 'ان': 'un', 'انھیں': 'unhein', 'انہیں': 'unhein', 'ہمیں': 'humein', 'تمہیں': 'tumhein',
    'मुझे': 'mujhe', 'اسے': 'use', 'کا': 'ka', 'کی': 'ki', 'کے': 'ke', 'کو': 'ko', 'سے': 'se',
    'پر': 'par', 'تک': 'tak', 'نے': 'ne', 'اور': 'aur', 'یا': 'ya', 'لیکن': 'lekin', 'مگر': 'magar',
    'اگر': 'agar', 'کیونکہ': 'kyunke', 'تو': 'toh', 'بھی': 'bhi', 'ہی': 'hi', 'ساتھ': 'saath',
    'کیا': 'kya', 'کون': 'kaun', 'کب': 'kab', 'کہاں': 'kahan', 'کیسے': 'kaise', 'کیوں': 'kyun',
    'کتنا': 'kitna', 'کس': 'kis', 'ہے': 'hai', 'ہیں': 'hain', 'ہو': 'ho', 'ہوں': 'hoon',
    'تھا': 'tha', 'تھی': 'thi', 'تھے': 'thay', 'کر': 'kar', 'کرنا': 'karna', 'رہا': 'raha',
    'رہی': 'rahi', 'رہے': 'rahe', 'ہوا': 'hua', 'ہوئی': 'hui', 'ہوئے': 'hue', 'نہیں': 'nahin',
    'صحیح': 'sahi', 'بہت': 'bohot', 'اچھا': 'acha', 'شکریہ': 'shukriya', 'سلام': 'salam',
    'پروجیکٹ': 'project', 'ٹیسٹ': 'test', 'کلاس': 'class', 'سٹوڈنٹ': 'student', 'یونیورسٹی': 'university'
}

def to_roman_urdu(text: str) -> str:
    if not text: return ""
    words = text.split()
    romanized = []
    for w in words:
        clean_w = re.sub(r'[\u064B-\u065F\u0670]', '', w)
        if clean_w in URDU_WORD_DICT:
            romanized.append(URDU_WORD_DICT[clean_w])
        else:
            romanized.append(w)
    return " ".join(romanized)

def sanitize_script(text: str) -> str:
    cleaned = re.sub(r'[\u0900-\u097F\u0F00-\u0FFF\u25A0-\u25FF\uFFFD]+', '', text)
    return cleaned.strip()

# ─── 5. Audio Processing Helpers ───
def load_16k_from_file(path):
    try:
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
        if sr != 16000: wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        d = wav.squeeze().numpy().astype(np.float32)
        return audio_cleaner.filter_noise(d)
    except Exception:
        d, sr = sf.read(path)
        if d.ndim > 1: d = d.mean(1)
        d = d.astype(np.float32)
        if sr != 16000:
            n = int(len(d)*16000/sr)
            d = np.interp(np.linspace(0,len(d),n,endpoint=False),np.arange(len(d)),d).astype(np.float32)
        return audio_cleaner.filter_noise(d)

def convert_chunk_to_16k(audio_chunk):
    if audio_chunk is None: return None
    sr, y = audio_chunk
    if y is None or len(y) == 0: return None
    if y.dtype == np.int16:
        y = y.astype(np.float32) / 32768.0
    elif y.dtype == np.int32:
        y = y.astype(np.float32) / 2147483648.0
    elif y.dtype != np.float32:
        y = y.astype(np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != 16000:
        n_samples = int(len(y) * 16000 / sr)
        y = np.interp(np.linspace(0, len(y), n_samples, endpoint=False), np.arange(len(y)), y).astype(np.float32)
    return audio_cleaner.filter_noise(y)

COLORS = ['#6366F1','#10B981','#F59E0B','#EC4899','#06B6D4','#8B5CF6','#F97316','#38BDF8']
MODEL_MAP = {
    'Large-v3-Turbo (809M)': 'large-v3-turbo',
    'Whisper Large-v3 (1.5B)': 'large-v3',
    'Whisper Base (74M)': 'base',
    'Whisper Tiny (39M)': 'tiny',
    'Whisper Small (244M)': 'small',
    'Whisper Medium (769M)': 'medium',
}
LANG_MAP = {
    'Bilingual (Urdu + English)': ('ur', False, False),
    'Pure Urdu Script (اردو)': ('ur', True, False),
    'English Only': ('en', False, True),
    'Roman Urdu (Latin)': ('ur', False, True),
}

PROMPT_MAP = {
    'Bilingual (Urdu + English)': "Bilingual Pakistani university classroom lecture in mixed English and Urdu. Discussion on computer science, concepts, code, formulas, assignments, presentations, questions and answers.",
    'Pure Urdu Script (اردو)': "یہ ایک پاکستانی یونیورسٹی کا کلاس روم لیکچر ہے۔ تمام الفاظ اور گفتگو کو خالص اور درست اردو رسم الخط میں تحریر کریں۔ انگریزی اصطلاحات کو بھی اردو رسم الخط میں لکھیں۔",
    'English Only': "University classroom lecture delivered strictly in English. Discussion on computer science, technical definitions, formulas, and academic questions.",
    'Roman Urdu (Latin)': "Pakistani university classroom lecture in Roman Urdu and English language. Code-switching discussion and concepts.",
}

# ─── 6. Vectorized Speaker Diarization Matcher ───
def identify_speaker(emb, duration, current_time, embeddings_dict, last_spk_id, last_spk_time, threshold=0.28):
    if duration < 1.4 and last_spk_id is not None:
        return last_spk_id, 0.99, last_spk_id, current_time

    if not embeddings_dict:
        new_id = 1
        embeddings_dict[new_id] = [emb]
        return new_id, 1.0, new_id, current_time

    best_id = None
    best_score = -1.0

    for sid, embs in embeddings_dict.items():
        mat = np.array(embs, dtype=np.float32)
        centroid = np.mean(mat, axis=0)
        c_norm = np.linalg.norm(centroid) or 1.0
        centroid = centroid / c_norm
        c_score = float(np.dot(centroid, emb))

        scores = np.dot(mat, emb)
        max_s = float(np.max(scores))
        k = min(3, len(scores))
        top_k = np.partition(scores, -k)[-k:]
        top_avg = float(np.mean(top_k))
        
        score = max(c_score, top_avg * 0.85 + max_s * 0.15)

        if last_spk_id == sid and (current_time - last_spk_time) < 15.0:
            score += 0.08

        if len(embeddings_dict) == 1 and sid == 1:
            score += 0.04

        if score > best_score:
            best_score = score
            best_id = sid

    effective_thresh = max(0.24, threshold)
    if best_id is not None and best_score >= effective_thresh:
        if len(embeddings_dict[best_id]) < 50:
            embeddings_dict[best_id].append(emb)
        return best_id, best_score, best_id, current_time
    else:
        new_id = len(embeddings_dict) + 1
        embeddings_dict[new_id] = [emb]
        return new_id, best_score, new_id, current_time

# ─── Render HTML & Direct Base64 Data URL Exports ───
def render_transcript_ui(segments_list, profiles_dict, spk_names_dict, mkey, elapsed_total):
    if not segments_list:
        empty_t = """<div class='vd-empty'>
          <svg width='48' height='48' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
            <path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/>
          </svg>
          <p>Lecture transcript will stream live here</p>
          <span>Speak into the microphone or upload an audio recording</span>
        </div>"""
        empty_s = """<div class='vd-empty-sm'>
          <svg width='32' height='32' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
            <path d='M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2'/><circle cx='9' cy='7' r='4'/>
            <path d='M23 21v-2a4 4 0 0 0-3-3.87'/><path d='M16 3.13a4 4 0 0 1 0 7.75'/>
          </svg>
          <p>No active speaker profiles</p>
          <span>Start recording to build neural voiceprints</span>
        </div>"""
        empty_e = """<div class="vd-export-grid">
          <span class="vd-dl disabled">Markdown (.md)</span>
          <span class="vd-dl disabled">Plain Text (.txt)</span>
          <span class="vd-dl disabled">Subtitles (.srt)</span>
          <span class="vd-dl disabled">JSON (.json)</span>
        </div>"""
        return empty_t, empty_s, empty_e, ""

    html_parts = []
    plain = []
    srt_parts = []
    json_arr = []
    si = 1

    for seg in segments_list:
        spk = seg["id"]
        display_name = spk_names_dict.get(spk, seg.get("speaker", f"Speaker {spk}"))
        c = COLORS[(spk-1) % len(COLORS)]
        ts = seg["time"]
        txt = seg["text"]
        urdu = any('\u0600' <= ch <= '\u06FF' for ch in txt)
        txt_class = "vd-txt vd-rtl" if urdu else "vd-txt"

        html_parts.append(f"""<div class="vd-seg" style="border-left-color:{c};">
  <div class="vd-seg-body">
    <div class="vd-seg-meta">
      <span class="vd-dot" style="background:{c};box-shadow:0 0 8px {c};"></span>
      <span class="vd-spk-lbl" style="color:{c};">{display_name}</span>
      <span class="vd-time">[{ts}]</span>
    </div>
    <div class="{txt_class}">{txt}</div>
  </div>
</div>""")
        plain.append(f"[{ts}] {display_name}: {txt}")

        def srt_ts(s):
            return f"{int(s//3600):02d}:{int(s%3600//60):02d}:{int(s%60):02d},{int((s-int(s))*1000):03d}"
        srt_parts.append(f"{si}\n{srt_ts(seg['start'])} --> {srt_ts(seg['end'])}\n[{display_name}]: {txt}\n")
        json_arr.append({"speaker": display_name, "id": spk, "start": round(seg['start'],2), "end": round(seg['end'],2), "time": ts, "text": txt})
        si += 1

    sb = []
    for sid, count in profiles_dict.items():
        cc = COLORS[(sid-1) % len(COLORS)]
        sname = spk_names_dict.get(sid, f"Speaker {sid}")
        initial = sname[0].upper() if sname else "S"
        sb.append(f"""<div class="vd-spk-card">
  <div class="vd-spk-av" style="background:{cc};box-shadow:0 0 12px {cc}55">{initial}</div>
  <div class="vd-spk-info">
    <div class="vd-spk-name">{sname}</div>
    <div class="vd-spk-meta">{count} centroid voice print{'s' if count!=1 else ''}</div>
  </div>
  <button class="vd-spk-edit-btn" onclick="renameSpeaker({sid}, '{sname}')" title="Rename {sname}">
    <svg width="13" height="13" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.2">
      <path d="M11 4H4a2 2 0 0 0-2 2v14a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2v-7"></path>
      <path d="M18.5 2.5a2.121 2.121 0 0 1 3 3L12 15l-4 1 1-4 9.5-9.5z"></path>
    </svg>
    <span>Edit</span>
  </button>
</div>""")
    if not sb:
        sb = ["<div class='vd-empty-sm'><p>No active speaker profiles</p></div>"]

    total_dur = segments_list[-1]["end"] if segments_list else 0.0
    stats = f"""<div class="vd-stats">
  <span>⚡ {mkey} · {gpu_name} ({compute_dtype.upper()})</span>
  <span>{len(segments_list)} segments · {total_dur:.1f}s recorded ({elapsed_total:.1f}s compute)</span>
</div>"""

    full_transcript = "\n".join(html_parts) + stats

    # Direct In-Browser Data URL Encoded Downloads
    md_content   = f"# VoiceDiary Lecture Notes\n**Model:** {mkey} | **Compute:** {gpu_name}\n\n" + "\n\n".join(plain)
    txt_content  = "\n".join(plain)
    srt_content  = "\n".join(srt_parts)
    json_content = json.dumps(json_arr, indent=2, ensure_ascii=False)

    b64_md   = base64.b64encode(md_content.encode('utf-8')).decode('ascii')
    b64_txt  = base64.b64encode(txt_content.encode('utf-8')).decode('ascii')
    b64_srt  = base64.b64encode(srt_content.encode('utf-8')).decode('ascii')
    b64_json = base64.b64encode(json_content.encode('utf-8')).decode('ascii')

    data_uri_md   = f"data:text/markdown;charset=utf-8;base64,{b64_md}"
    data_uri_txt  = f"data:text/plain;charset=utf-8;base64,{b64_txt}"
    data_uri_srt  = f"data:text/plain;charset=utf-8;base64,{b64_srt}"
    data_uri_json = f"data:application/json;charset=utf-8;base64,{b64_json}"

    export_html = f"""<div class="vd-export-grid">
  <a class="vd-dl" href="{data_uri_md}" download="VoiceDiary_Lecture_Notes.md">
    <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/><line x1="12" y1="18" x2="12" y2="12"/><line x1="9" y1="15" x2="15" y2="15"/></svg>
    Markdown (.md)
  </a>
  <a class="vd-dl" href="{data_uri_txt}" download="VoiceDiary_Transcript.txt">
    <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8z"/><polyline points="14 2 14 8 20 8"/></svg>
    Plain Text (.txt)
  </a>
  <a class="vd-dl" href="{data_uri_srt}" download="VoiceDiary_Subtitles.srt">
    <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="2" y="2" width="20" height="20" rx="2"/><path d="M8 10h8M8 14h5"/></svg>
    Subtitles (.srt)
  </a>
  <a class="vd-dl" href="{data_uri_json}" download="VoiceDiary_Lecture_Data.json">
    <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><polyline points="16 18 22 12 16 6"/><polyline points="8 6 2 12 8 18"/></svg>
    JSON Data (.json)
  </a>
</div>"""

    return full_transcript, "\n".join(sb), export_html, "\n".join(plain)

# ─── Gemini 2.5 Flash Summarizer ───
def gemini_summary(text, key):
    if not text or not text.strip():
        return "⚠️ *Please transcribe a lecture first before generating an AI summary.*"
    api_key = (key or "").strip() or os.environ.get("GEMINI_API_KEY","")
    if not api_key:
        return "⚠️ *Please enter your Gemini API Key in the left sidebar to generate structured study notes.*"
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    prompt = (
        f"You are VoiceDiary AI, an elite university academic note-taker. "
        f"Analyze this classroom lecture transcript (mixed English and Urdu) "
        f"and produce clean, beautifully structured study notes in standard Markdown.\n\n"
        f"CRITICAL FORMATTING GUIDELINES:\n"
        f"- Start directly with the lecture title: # [Lecture Title]\n"
        f"- Do NOT include conversational filler (do NOT say 'Here are your notes', do NOT write horizontal rules '---' under title).\n"
        f"- Never write negative disclaimers like '(No formulas were mentioned)'. Focus purely on taught concepts.\n"
        f"- Format Study Flashcards clearly as:\n"
        f"  * **Q:** [Question]?\n"
        f"    **A:** [Clear concise answer]\n\n"
        f"REQUIRED SECTIONS:\n"
        f"## 📌 Executive Lecture Summary\n"
        f"(3-4 high-yield bullet points summarizing core themes)\n\n"
        f"## 🎯 Key Academic Concepts & Definitions\n"
        f"(In-depth breakdown of concepts, formulas, logic, or technical steps discussed)\n\n"
        f"## 💡 Study Flashcards & Exam Revision\n"
        f"(4-5 high-yield Question and Answer pairs for rapid exam revision)\n\n"
        f"## 📋 Action Items & Homework Mentioned\n"
        f"(Any tasks, assignments, deadlines, or follow-ups mentioned, or 'None mentioned' if none)\n\n"
        f"Transcript:\n{text}"
    )
    payload = {"contents":[{"parts":[{"text": prompt}]}]}
    try:
        req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                     headers={"Content-Type":"application/json"})
        with urllib.request.urlopen(req, timeout=35) as r:
            res = json.loads(r.read())
            raw = res["candidates"][0]["content"]["parts"][0]["text"]
            clean_md = re.sub(r'^(?:Here are (?:the|your)|Sure,|Below is|Here is|Note:).*?$\n?', '', raw.strip(), flags=re.MULTILINE | re.IGNORECASE)
            clean_md = re.sub(r'\(No (?:mathematical|formulas|code).*?\)', '', clean_md, flags=re.IGNORECASE)
            clean_md = re.sub(r'(#[^\n]+\n+)\s*---\s*\n+', r'\1', clean_md)
            return clean_md.strip()
    except Exception as e:
        return f"❌ *Gemini AI Error: {e}*"

# ─── Symmetrical Architectural CSS ───
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Fira+Code:wght@400;500;600&family=Noto+Nastaliq+Urdu:wght@400;700&display=swap');

html, body { margin: 0 !important; padding: 0 !important; height: 100% !important; background: #080C14 !important; }

.gradio-container {
  max-width: 100% !important;
  width: 100% !important;
  min-height: 100vh !important;
  margin: 0 !important;
  padding: 0 28px 36px 28px !important;
  background: #080C14 !important;
  font-family: 'Plus Jakarta Sans', -apple-system, BlinkMacSystemFont, sans-serif !important;
  color: #F8FAFC !important;
}

.gradio-container::before {
  content: '';
  position: fixed; inset: 0; z-index: 0; pointer-events: none;
  background:
    radial-gradient(ellipse 80% 45% at 50% -10%, rgba(99,102,241,.16) 0%, transparent 65%),
    radial-gradient(ellipse 60% 35% at 85% 90%, rgba(139,92,246,.08) 0%, transparent 55%);
}

/* Hidden DOM Bridge for Speaker Rename */
.vd-hidden-bridge,
#hidden_rename_payload,
#hidden_rename_trigger {
  position: absolute !important;
  opacity: 0 !important;
  pointer-events: none !important;
  width: 1px !important;
  height: 1px !important;
  overflow: hidden !important;
  clip: rect(0, 0, 0, 0) !important;
  z-index: -9999 !important;
  margin: 0 !important;
  padding: 0 !important;
}

/* Eliminate All Flickering */
.pending, .loading, .generating,
.gradio-container .pending,
.gradio-container [data-testid="loading"],
.gradio-container .loading {
  opacity: 1 !important;
  filter: none !important;
  transition: none !important;
  animation: none !important;
}
.loading-status, .progress-bar, .meta-text, .eta-bar,
.gradio-container .generating-status {
  display: none !important;
  visibility: hidden !important;
  opacity: 0 !important;
}

/* Nuke Audio Preview Pill & Floating Clear/X Box */
.gradio-container .audio [data-testid="audio-time"],
.gradio-container .audio .timestamps,
.gradio-container .audio select,
.gradio-container .audio .source-select,
.gradio-container .audio .player,
.gradio-container .audio .audio-player,
.gradio-container .audio [data-testid="audio-player"],
.gradio-container .audio [data-testid="clear-button"],
.gradio-container .audio .clear-button,
.gradio-container .audio button[aria-label="Clear"],
.gradio-container .audio button[title="Clear"],
.gradio-container .audio button.delete,
.gradio-container .audio button.icon-button,
.gradio-container .audio .waveform,
.gradio-container .audio .waveform-container,
.gradio-container .audio .controls,
.gradio-container .audio .action-buttons,
.gradio-container .audio .secondary-controls,
.gradio-container .audio .meta,
.gradio-container .audio audio,
.gradio-container .audio div:has(> button[aria-label="Clear"]),
.gradio-container .audio div:has(> button.delete) {
  display: none !important;
  visibility: hidden !important;
  opacity: 0 !important;
  height: 0 !important;
  width: 0 !important;
  margin: 0 !important;
  padding: 0 !important;
  position: absolute !important;
  pointer-events: none !important;
}

.gradio-container .audio button.record-button,
.gradio-container .audio button.primary,
.gradio-container .audio button:not([aria-label="Clear"]):not(.delete):not(.clear-button) {
  display: inline-flex !important;
  visibility: visible !important;
  opacity: 1 !important;
  position: relative !important;
}

/* Layout Structural Symmetry */
.gradio-container .block,
.gradio-container .form,
.gradio-container .gap {
  box-shadow: none !important;
  border: none !important;
  background: transparent !important;
  padding: 0 !important;
  margin: 0 !important;
}
.gr-group, .gr-box, .gr-panel { background: transparent !important; border: none !important; box-shadow: none !important; }
footer, .footer, .gr-footer { display: none !important; }

/* Unified Input & Dropdown Styling (100% Width & Matching 12px Radius) */
.gradio-container select,
.gradio-container input[type=text],
.gradio-container input[type=password],
.gradio-container textarea {
  width: 100% !important;
  box-sizing: border-box !important;
  background: rgba(15,23,42,.75) !important;
  border: 1px solid rgba(255,255,255,.10) !important;
  border-radius: 12px !important;
  color: #F8FAFC !important;
  font-family: inherit !important;
  font-size: 13.5px !important;
  font-weight: 500 !important;
  padding: 11px 14px !important;
  transition: all .15s ease !important;
}
.gradio-container select:focus,
.gradio-container input:focus {
  border-color: #6366F1 !important;
  box-shadow: 0 0 14px rgba(99,102,241,.30) !important;
  outline: none !important;
}

.gradio-container label > span,
.gradio-container .label-wrap span {
  color: #94A3B8 !important;
  font-size: 11px !important;
  font-weight: 700 !important;
  letter-spacing: .06em !important;
  text-transform: uppercase !important;
  margin-bottom: 6px !important;
  display: inline-block !important;
}

.gradio-container .gr-slider {
  margin-bottom: 12px !important;
}
.gradio-container .gr-slider input[type=number] {
  background: rgba(15,23,42,.85) !important;
  border: 1px solid rgba(255,255,255,.12) !important;
  color: #F8FAFC !important;
  border-radius: 8px !important;
  font-weight: 600 !important;
  font-size: 12.5px !important;
  padding: 4px 8px !important;
}

.tab-nav {
  background: transparent !important;
  border-bottom: 1px solid rgba(255,255,255,.08) !important;
  margin-bottom: 18px !important;
}
.tab-nav button {
  color: #94A3B8 !important; font-weight: 700 !important; font-size: 13.5px !important;
  padding: 11px 22px !important; background: transparent !important;
  border: none !important; border-bottom: 2px solid transparent !important;
  border-radius: 0 !important; transition: all .15s !important;
}
.tab-nav button.selected {
  color: #FFFFFF !important;
  border-bottom-color: #6366F1 !important;
  background: rgba(99,102,241,.08) !important;
}
.tabitem { background: transparent !important; border: none !important; padding: 0 !important; }

.gradio-container .audio,
.gradio-container .gr-audio {
  background: rgba(15,23,42,.70) !important;
  border: 1px solid rgba(255,255,255,.10) !important;
  border-radius: 14px !important;
  min-height: 80px !important;
  padding: 12px 16px !important;
  margin-bottom: 10px !important;
}

.gradio-container button.primary,
.vd-btn-primary {
  background: linear-gradient(135deg,#6366F1 0%,#8B5CF6 100%) !important;
  color: #FFFFFF !important; border: none !important;
  font-weight: 700 !important; font-size: 14.5px !important;
  border-radius: 12px !important; padding: 13px 26px !important;
  box-shadow: 0 0 20px rgba(99,102,241,.30) !important;
  transition: all .2s !important; width: 100% !important; cursor: pointer !important;
}
.gradio-container button.primary:hover,
.vd-btn-primary:hover {
  transform: translateY(-1px) !important;
  box-shadow: 0 0 28px rgba(99,102,241,.50) !important;
}

.gradio-container button.secondary {
  background: rgba(255,255,255,.05) !important;
  border: 1px solid rgba(255,255,255,.10) !important;
  color: #E2E8F0 !important; font-weight: 600 !important; font-size: 12.5px !important;
  border-radius: 10px !important; padding: 8px 16px !important; transition: all .15s !important;
}
.gradio-container button.secondary:hover { background: rgba(255,255,255,.10) !important; color: #FFFFFF !important; }

.vd-hdr {
  display: flex; align-items: center; justify-content: space-between;
  padding: 18px 0; margin-bottom: 22px;
  border-bottom: 1px solid rgba(255,255,255,.08);
}
.vd-hdr-left { display: flex; align-items: center; gap: 14px; }
.vd-logo-box {
  width: 42px; height: 42px; border-radius: 12px; flex-shrink: 0;
  background: linear-gradient(135deg,#6366F1,#8B5CF6);
  display: flex; align-items: center; justify-content: center;
  box-shadow: 0 0 18px rgba(99,102,241,.40);
}
.vd-hdr-title { font-size: 21px; font-weight: 800; color: #FFFFFF; letter-spacing: -.02em; }
.vd-hdr-sub { font-size: 12.5px; color: #94A3B8; font-weight: 500; margin-top: 2px; }
.vd-hw-pill {
  display: inline-flex; align-items: center; gap: 8px;
  padding: 6px 16px; border-radius: 9999px;
  background: rgba(16,185,129,.10); border: 1px solid rgba(16,185,129,.24);
  font-size: 11.5px; font-weight: 700; color: #10B981;
  font-family: 'Fira Code', monospace;
}
.vd-hw-dot { width: 7px; height: 7px; border-radius: 50%; background: #10B981; box-shadow: 0 0 8px rgba(16,185,129,.8); }

.vd-sec-lbl {
  font-size: 11px; font-weight: 800; color: #94A3B8;
  letter-spacing: .08em; text-transform: uppercase;
  display: flex; align-items: center; justify-content: space-between;
  margin-bottom: 10px; margin-top: 0;
}
.vd-badge-live {
  width: 7px; height: 7px; border-radius: 50%; background: #10B981;
  box-shadow: 0 0 8px rgba(16,185,129,.9); display: inline-block; margin-right: 5px;
  animation: pulse 2s infinite;
}
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:.4} }

.vd-spk-card {
  display: flex; align-items: center; gap: 12px;
  padding: 11px 14px; border-radius: 12px;
  background: rgba(15,23,42,.70); border: 1px solid rgba(255,255,255,.08);
  margin-bottom: 8px; transition: all .15s ease;
  box-sizing: border-box; width: 100%;
}
.vd-spk-card:hover {
  background: rgba(30,41,59,.80);
  border-color: rgba(99,102,241,.35);
  transform: translateX(2px);
}
.vd-spk-av {
  width: 36px; height: 36px; border-radius: 50%; flex-shrink: 0;
  display: flex; align-items: center; justify-content: center;
  font-weight: 800; font-size: 13.5px; color: #FFFFFF;
}
.vd-spk-info { flex: 1; min-width: 0; }
.vd-spk-name { font-size: 13.5px; font-weight: 700; color: #F1F5F9; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.vd-spk-meta { font-size: 11px; color: #64748B; margin-top: 2px; }

.vd-spk-edit-btn {
  display: inline-flex !important;
  align-items: center !important;
  gap: 5px !important;
  padding: 5px 11px !important;
  border-radius: 8px !important;
  background: rgba(255,255,255,.06) !important;
  border: 1px solid rgba(255,255,255,.12) !important;
  color: #94A3B8 !important;
  font-size: 11.5px !important;
  font-weight: 600 !important;
  cursor: pointer !important;
  transition: all .15s ease !important;
  flex-shrink: 0 !important;
}
.vd-spk-edit-btn:hover {
  background: rgba(99,102,241,.25) !important;
  border-color: rgba(99,102,241,.6) !important;
  color: #FFFFFF !important;
}

.vd-transcript-vp {
  background: rgba(10,15,28,.85); border: 1px solid rgba(255,255,255,.08);
  border-radius: 14px; padding: 20px; min-height: 380px; max-height: 520px;
  overflow-y: auto; backdrop-filter: blur(16px);
}
.vd-seg {
  margin-bottom: 12px; border-radius: 12px;
  background: rgba(15,23,42,.70); border: 1px solid rgba(255,255,255,.08);
  border-left-width: 4px; border-left-style: solid;
  overflow: hidden; transition: .15s ease;
}
.vd-seg:hover { background: rgba(30,41,59,.80); border-color: rgba(255,255,255,.14); }
.vd-seg-body { padding: 14px 18px; }
.vd-seg-meta { display: flex; align-items: center; gap: 10px; margin-bottom: 6px; }
.vd-dot { width: 7px; height: 7px; border-radius: 50%; display: inline-block; }
.vd-spk-lbl { font-size: 13.5px; font-weight: 700; }
.vd-time { font-size: 11.5px; color: #64748B; font-family: 'Fira Code', monospace; }
.vd-txt { font-size: 15px; line-height: 1.65; color: #F1F5F9; word-break: break-word; font-weight: 400; }
.vd-rtl { direction: rtl; text-align: right; font-family: 'Noto Nastaliq Urdu', serif; font-size: 19px; line-height: 2.1; color: #FFFFFF; }

.vd-stats {
  margin-top: 14px; padding-top: 12px; border-top: 1px solid rgba(255,255,255,.08);
  display: flex; justify-content: space-between; align-items: center;
  font-size: 11.5px; color: #64748B; font-family: 'Fira Code', monospace;
}

.vd-empty {
  display: flex; flex-direction: column; align-items: center; justify-content: center;
  padding: 70px 20px; text-align: center; color: #475569; gap: 10px;
}
.vd-empty p { font-size: 16px; font-weight: 700; color: #94A3B8; margin: 0; }
.vd-empty span { font-size: 12.5px; color: #64748B; }
.vd-empty-sm {
  display: flex; flex-direction: column; align-items: center;
  padding: 20px 10px; text-align: center; color: #475569; gap: 6px;
}
.vd-empty-sm p { font-size: 12.5px; font-weight: 600; color: #94A3B8; margin: 0; }
.vd-empty-sm span { font-size: 11px; color: #64748B; }

.vd-export-grid {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 10px;
  margin-top: 6px;
  width: 100%;
}
@media (max-width: 900px) {
  .vd-export-grid { grid-template-columns: repeat(2, 1fr); }
}

.vd-dl {
  display: inline-flex !important; align-items: center !important; justify-content: center !important;
  gap: 7px !important; padding: 10px 14px !important; border-radius: 10px !important;
  background: rgba(255,255,255,.05) !important; border: 1px solid rgba(255,255,255,.10) !important;
  color: #E2E8F0 !important; font-size: 13px !important; font-weight: 600 !important;
  text-decoration: none !important; transition: all .15s ease !important; cursor: pointer !important;
  width: 100% !important; box-sizing: border-box !important;
}
.vd-dl:hover {
  background: rgba(99,102,241,.18) !important;
  border-color: rgba(99,102,241,.45) !important;
  color: #A5B4FC !important;
  transform: translateY(-1px) !important;
}
.vd-dl.disabled { opacity: 0.4 !important; cursor: not-allowed !important; pointer-events: none !important; }

.vd-divider {
  border: none !important;
  border-top: 1px solid rgba(255,255,255,.08) !important;
  margin: 18px 0 !important;
}

.vd-summary-card {
  background: rgba(15,23,42,.70) !important;
  border: 1px solid rgba(255,255,255,.10) !important;
  border-radius: 14px !important;
  padding: 22px !important;
  font-size: 14.5px !important;
  line-height: 1.7 !important;
  color: #F1F5F9 !important;
  backdrop-filter: blur(16px) !important;
  margin-top: 12px !important;
}

.vd-ftr {
  margin-top: 36px !important;
  padding: 22px 0 12px 0 !important;
  border-top: 1px solid rgba(255,255,255,.08) !important;
  text-align: center !important;
  color: #64748B !important;
  font-size: 13px !important;
  font-weight: 500 !important;
}
.vd-ftr-content {
  display: flex !important;
  align-items: center !important;
  justify-content: center !important;
  gap: 10px !important;
  flex-wrap: wrap !important;
}
.vd-ftr strong {
  color: #F1F5F9 !important;
  font-weight: 700 !important;
}
.vd-ftr-dot {
  color: #475569 !important;
  font-weight: 800 !important;
}
"""

HEAD_JS = """
<script>
window.renameSpeaker = function(spkId, oldName) {
  const newName = prompt(`Enter new display name for Speaker ${spkId}:`, oldName);
  if (newName !== null && newName.trim() !== "" && newName.trim() !== oldName) {
    const input = document.querySelector('#hidden_rename_payload textarea, #hidden_rename_payload input');
    const btn = document.querySelector('#hidden_rename_trigger');
    if (input && btn) {
      const val = JSON.stringify({id: parseInt(spkId), name: newName.trim()});
      const setter = Object.getOwnPropertyDescriptor(window.HTMLTextAreaElement ? window.HTMLTextAreaElement.prototype : window.HTMLInputElement.prototype, 'value') || Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype, 'value');
      if (setter && setter.set) {
        setter.set.call(input, val);
      } else {
        input.value = val;
      }
      input.dispatchEvent(new Event('input', { bubbles: true, composed: true }));
      input.dispatchEvent(new Event('change', { bubbles: true, composed: true }));
      setTimeout(() => { btn.click(); }, 60);
    }
  }
};
</script>
"""

with gr.Blocks(title="VoiceDiary — AI Bilingual Lecture & Diarization Engine", css=CSS, head=HEAD_JS,
               theme=gr.themes.Default(primary_hue="indigo", neutral_hue="slate")) as demo:

    # Global Session State
    transcript_state = gr.State("")
    session_segments_state = gr.State([])
    speaker_profiles_state = gr.State({})
    speaker_names_state = gr.State({})
    speaker_embeddings_state = gr.State({})
    last_speaker_id_state = gr.State(None)
    last_speaker_time_state = gr.State(0.0)
    
    # Adaptive VAD Pause-Triggered Streaming Buffer State
    speech_accumulator = gr.State(np.array([], dtype=np.float32))
    is_speaking_state = gr.State(False)
    silence_counter_state = gr.State(0.0)
    total_timeline_sec = gr.State(0.0)
    total_compute_time = gr.State(0.0)

    # ── HEADER ──
    gr.HTML(f"""
    <div class="vd-hdr">
      <div class="vd-hdr-left">
        <div class="vd-logo-box">
          <svg width="20" height="20" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="2.2">
            <path d="M12 1a3 3 0 0 0-3 3v8a3 3 0 0 0 6 0V4a3 3 0 0 0-3-3z"/>
            <path d="M19 10v2a7 7 0 0 1-14 0v-2"/>
            <line x1="12" y1="19" x2="12" y2="23"/><line x1="8" y1="23" x2="16" y2="23"/>
          </svg>
        </div>
        <div>
          <div class="vd-hdr-title">VoiceDiary</div>
          <div class="vd-hdr-sub">Bilingual Classroom Lecture Note-Taking &amp; Neural Diarization Engine</div>
        </div>
      </div>
      <div class="vd-hw-pill">
        <span class="vd-hw-dot"></span>
        {gpu_name} &nbsp;·&nbsp; Tensor Cores {compute_dtype.upper()}
      </div>
    </div>""")

    # Mounted DOM Bridge for Native Speaker Rename
    hidden_rename_payload = gr.Textbox(elem_id="hidden_rename_payload", elem_classes=["vd-hidden-bridge"])
    hidden_rename_trigger = gr.Button("Trigger Rename", elem_id="hidden_rename_trigger", elem_classes=["vd-hidden-bridge"])

    with gr.Row(equal_height=False):
        # ── LEFT SIDEBAR ──
        with gr.Column(scale=3, min_width=280):
            gr.HTML("<div class='vd-sec-lbl'><span>Speakers &amp; Neural Profiles</span><span><span class='vd-badge-live'></span>LIVE</span></div>")
            sidebar_out = gr.HTML(value="""<div class='vd-empty-sm'>
              <svg width='32' height='32' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
                <path d='M17 21v-2a4 4 0 0 0-4-4H5a4 4 0 0 0-4 4v2'/><circle cx='9' cy='7' r='4'/>
                <path d='M23 21v-2a4 4 0 0 0-3-3.87'/><path d='M16 3.13a4 4 0 0 1 0 7.75'/>
              </svg>
              <p>No active speaker profiles</p>
              <span>Start recording to build neural voiceprints</span>
            </div>""")

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>AI Engine &amp; Whisper Model</span></div>")
            model_dd = gr.Dropdown(choices=list(MODEL_MAP.keys()),
                value='Large-v3-Turbo (809M)', label='Active Whisper Model', container=False)
            gr.HTML("<div style='height:8px;'></div>")
            lang_dd = gr.Dropdown(choices=list(LANG_MAP.keys()),
                value='Bilingual (Urdu + English)', label='Language Output Mode', container=False)

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Diarization &amp; Noise Sensitivity</span></div>")
            thresh_sl = gr.Slider(0, 100, 30, step=1, label='Speaker Match Threshold (%)')
            vad_sl = gr.Slider(0, 600, 300, step=10, label='Pause Trigger Gap (ms)')

            gr.HTML("<hr class='vd-divider'>")
            gr.HTML("<div class='vd-sec-lbl'><span>Google Gemini AI (BYOK)</span></div>")
            gemini_key = gr.Textbox(placeholder='Paste Gemini API Key (AIzaSy…)', type='password',
                                    label='Gemini API Key', container=False)

        # ── RIGHT MAIN WORKSPACE ──
        with gr.Column(scale=9, min_width=520):
            with gr.Tabs():
                with gr.TabItem("🎙️ Live Classroom Microphone"):
                    audio_mic = gr.Audio(sources=["microphone"], type="numpy", streaming=True,
                                         label="Microphone", show_label=False)
                    clear_live_btn = gr.Button("Clear Live Session", variant="secondary", size="sm")
                with gr.TabItem("📁 Upload Pre-Recorded Lecture"):
                    audio_file = gr.Audio(sources=["upload"], type="filepath",
                                          label="Upload classroom audio (.wav, .mp3, .m4a, .flac)",
                                          show_label=False)
                    transcribe_file_btn = gr.Button("⚡ Transcribe & Diarize Uploaded Audio (GPU)",
                                                    variant="primary", elem_classes=["vd-btn-primary"])

            gr.HTML("<div class='vd-sec-lbl' style='margin-top:20px;'><span>Classroom Lecture Transcript</span></div>")
            transcript_out = gr.HTML(
                value="""<div class='vd-empty'>
                  <svg width='48' height='48' viewBox='0 0 24 24' fill='none' stroke='currentColor' stroke-width='1.5' opacity='.35'>
                    <path d='M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z'/>
                  </svg>
                  <p>Lecture transcript will stream live here</p>
                  <span>Speak into the microphone or upload audio</span>
                </div>""",
                elem_classes=["vd-transcript-vp"])

            gr.HTML("<div class='vd-sec-lbl' style='margin-top:20px;'><span>Export Lecture Notes</span></div>")
            export_html_out = gr.HTML(value="""<div class="vd-export-grid">
              <span class="vd-dl disabled">Markdown (.md)</span>
              <span class="vd-dl disabled">Plain Text (.txt)</span>
              <span class="vd-dl disabled">Subtitles (.srt)</span>
              <span class="vd-dl disabled">JSON (.json)</span>
            </div>""")

            gr.HTML("<hr class='vd-divider' style='margin:22px 0 18px 0;'>")
            gr.HTML("<div class='vd-sec-lbl'><span>AI Study Summary &amp; Flashcards (Gemini 2.5 Flash)</span></div>")
            ai_btn = gr.Button("✨ Generate AI Lecture Summary & Study Guide",
                               variant="primary", elem_classes=["vd-btn-primary"])
            ai_out = gr.Markdown(value="*AI study summary and key concepts will be generated here after clicking above.*",
                                 elem_classes=["vd-summary-card"])

    # ── FOOTER ──
    gr.HTML("""
    <div class="vd-ftr">
      <div class="vd-ftr-content">
        <span>VoiceDiary &copy; 2026 <strong>Abdul Sarim Khan</strong>. All Rights Reserved.</span>
        <span class="vd-ftr-dot">&middot;</span>
        <span>Bilingual AI Lecture Note-Taking &amp; Neural Diarization Engine</span>
      </div>
    </div>""")

    # ── EVENT WIRINGS ──
    
    # ── Universal Noise-Cleaned Live Streaming Processor ──
    def handle_audio_stream(chunk, speech_buf, in_speech, silence_dur, timeline_sec, segments_list, profiles_dict, names_dict, embeddings_dict,
                            last_spk_id, last_spk_time, compute_time, model_choice, lang_choice, thresh_pct, vad_ms):
        if chunk is None:
            return speech_buf, in_speech, silence_dur, timeline_sec, segments_list, profiles_dict, names_dict, embeddings_dict, last_spk_id, last_spk_time, compute_time, gr.skip(), gr.skip(), gr.skip(), gr.skip()

        # Step 1: 80Hz Low-Cut Acoustic Filter + 16kHz resampling
        y = convert_chunk_to_16k(chunk)
        if y is None or len(y) == 0:
            return speech_buf, in_speech, silence_dur, timeline_sec, segments_list, profiles_dict, names_dict, embeddings_dict, last_spk_id, last_spk_time, compute_time, gr.skip(), gr.skip(), gr.skip(), gr.skip()

        chunk_dur = len(y) / 16000.0
        timeline_sec += chunk_dur

        # Step 2: Universal Silero Neural VAD + Ambient Noise Gate (runs for ALL models)
        has_speech = audio_cleaner.is_speech_active(y, threshold=0.40)

        should_cut = False
        pause_gap_target = max(0.6, float(vad_ms) / 1000.0) if vad_ms else 0.90

        if has_speech:
            if not in_speech:
                in_speech = True
            if speech_buf is None or len(speech_buf) == 0:
                speech_buf = y
            else:
                speech_buf = np.append(speech_buf, y)
            silence_dur = 0.0

            if len(speech_buf) >= int(16000 * 14.0):
                should_cut = True
                in_speech = False
                silence_dur = 0.0
        else:
            if in_speech:
                if silence_dur < 0.3:
                    speech_buf = np.append(speech_buf, y)
                silence_dur += chunk_dur
                
                if silence_dur >= pause_gap_target:
                    should_cut = True
                    in_speech = False
                    silence_dur = 0.0

        if should_cut and speech_buf is not None and len(speech_buf) >= int(16000 * 0.8):
            utterance_audio = speech_buf.copy()
            speech_buf = np.array([], dtype=np.float32)

            t0 = time.time()
            mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
            model = get_model(mkey)
            target_lang, is_urdu_script, is_english_only = LANG_MAP.get(lang_choice, ('ur', False, False))
            is_roman = (lang_choice == 'Roman Urdu (Latin)')
            thresh = float(thresh_pct) / 100.0
            active_prompt = PROMPT_MAP.get(lang_choice, PROMPT_MAP['Bilingual (Urdu + English)'])

            try:
                segs, info = model.transcribe(
                    utterance_audio,
                    language=target_lang,
                    beam_size=1, best_of=1, temperature=0.0,
                    initial_prompt=active_prompt,
                    condition_on_previous_text=False,
                    without_timestamps=False,
                    no_speech_threshold=0.6,
                    compression_ratio_threshold=2.4,
                    vad_filter=True,
                    vad_parameters=dict(min_silence_duration_ms=int(vad_ms) if vad_ms else 300)
                )

                valid_words = []
                for s in segs:
                    raw = s.text.strip()
                    if not raw: continue
                    conf = float(getattr(s, "avg_logprob", getattr(s, "avg_log_prob", 0.0)))
                    no_speech_prob = float(getattr(s, "no_speech_prob", 0.0))
                    if no_speech_prob > 0.65 or conf < -1.2: continue
                    raw = re.sub(r'\b(\w+(?:\s+\w+){0,3})(?:\s+\1\b)+', r'\1', raw, flags=re.IGNORECASE).strip()
                    raw = sanitize_script(raw)
                    if not raw or len(raw) < 2: continue
                    valid_words.append((s, raw))

                if valid_words:
                    combined_text_list = []
                    for s, raw in valid_words:
                        if is_urdu_script:
                            txt_clean = UrduNormalizer.normalize(raw)
                        elif is_english_only:
                            txt_clean = post_process_english(raw)
                        elif is_roman:
                            norm_ur = UrduNormalizer.normalize(raw)
                            txt_clean = post_process_english(to_roman_urdu(norm_ur))
                        else:
                            words = raw.split()
                            proc = []
                            for w in words:
                                if any('\u0600' <= ch <= '\u06FF' for ch in w):
                                    proc.append(UrduNormalizer.normalize(w))
                                else:
                                    proc.append(w)
                            txt_clean = post_process_english(" ".join(proc))
                        if txt_clean and len(txt_clean.strip()) > 0:
                            combined_text_list.append(txt_clean)

                    full_sentence = " ".join(combined_text_list).strip()
                    
                    if full_sentence and len(full_sentence) > 0:
                        spk = 1
                        slice_duration = len(utterance_audio) / 16000.0
                        try:
                            with torch.inference_mode():
                                w_t = torch.from_numpy(utterance_audio).float().unsqueeze(0).to(device_type)
                                e = embedder.encode_batch(w_t).squeeze().detach().cpu().numpy()
                                en = e / (np.linalg.norm(e) or 1.)

                            spk, score, last_spk_id, last_spk_time = identify_speaker(
                                en, slice_duration, timeline_sec, embeddings_dict, last_spk_id, last_spk_time, threshold=thresh
                            )
                            profiles_dict[spk] = len(embeddings_dict.get(spk, [1]))
                        except Exception as em_err:
                            print("Embedding error:", em_err)

                        if spk not in names_dict:
                            names_dict[spk] = f"Speaker {spk}"

                        seg_start = max(0.0, timeline_sec - slice_duration)
                        ts = f"{int(seg_start//60):02d}:{int(seg_start%60):02d}"

                        segments_list.append({
                            "speaker": names_dict.get(spk, f"Speaker {spk}"),
                            "id": spk,
                            "start": seg_start,
                            "end": timeline_sec,
                            "time": ts,
                            "text": full_sentence
                        })
            except Exception as e:
                print("Stream transcription error:", e)

            compute_time += (time.time() - t0)
            t_html, s_html, exp_html, plain_txt = render_transcript_ui(
                segments_list, profiles_dict, names_dict, mkey, compute_time
            )
            return speech_buf, in_speech, silence_dur, timeline_sec, segments_list, profiles_dict, names_dict, embeddings_dict, last_spk_id, last_spk_time, compute_time, t_html, s_html, exp_html, plain_txt

        return speech_buf, in_speech, silence_dur, timeline_sec, segments_list, profiles_dict, names_dict, embeddings_dict, last_spk_id, last_spk_time, compute_time, gr.skip(), gr.skip(), gr.skip(), gr.skip()

    audio_mic.stream(
        fn=handle_audio_stream,
        inputs=[audio_mic, speech_accumulator, is_speaking_state, silence_counter_state, total_timeline_sec, session_segments_state, speaker_profiles_state, speaker_names_state, speaker_embeddings_state,
                last_speaker_id_state, last_speaker_time_state, total_compute_time,
                model_dd, lang_dd, thresh_sl, vad_sl],
        outputs=[speech_accumulator, is_speaking_state, silence_counter_state, total_timeline_sec, session_segments_state, speaker_profiles_state, speaker_names_state, speaker_embeddings_state,
                 last_speaker_id_state, last_speaker_time_state, total_compute_time,
                 transcript_out, sidebar_out, export_html_out, transcript_state],
        show_progress="hidden"
    )

    def reset_live_session(model_choice):
        mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
        empty_t, empty_s, empty_e, _ = render_transcript_ui([], {}, {}, mkey, 0.0)
        return (
            np.array([], dtype=np.float32), False, 0.0, 0.0, [], {}, {}, {}, None, 0.0, 0.0,
            empty_t, empty_s, empty_e, ""
        )
    
    clear_live_btn.click(
        fn=reset_live_session,
        inputs=[model_dd],
        outputs=[speech_accumulator, is_speaking_state, silence_counter_state, total_timeline_sec, session_segments_state, speaker_profiles_state, speaker_names_state, speaker_embeddings_state,
                 last_speaker_id_state, last_speaker_time_state, total_compute_time,
                 transcript_out, sidebar_out, export_html_out, transcript_state],
        show_progress="hidden"
    )

    def handle_inline_rename(payload_json, segments_list, profiles_dict, names_dict, compute_time, model_choice):
        if payload_json:
            try:
                data = json.loads(payload_json)
                sid = int(data.get("id", 1))
                new_name = str(data.get("name", "")).strip()
                if new_name:
                    names_dict[sid] = new_name
                    for seg in segments_list:
                        if seg["id"] == sid:
                            seg["speaker"] = new_name
            except Exception as e:
                print("Rename error:", e)

        mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
        t_html, s_html, exp_html, plain_txt = render_transcript_ui(
            segments_list, profiles_dict, names_dict, mkey, compute_time
        )
        return names_dict, segments_list, t_html, s_html, exp_html, plain_txt

    hidden_rename_trigger.click(
        fn=handle_inline_rename,
        inputs=[hidden_rename_payload, session_segments_state, speaker_profiles_state, speaker_names_state, total_compute_time, model_dd],
        outputs=[speaker_names_state, session_segments_state, transcript_out, sidebar_out, export_html_out, transcript_state],
        show_progress="hidden"
    )

    def handle_file_upload(fpath, model_choice, lang_choice, thresh_pct, vad_ms):
        mkey = MODEL_MAP.get(model_choice, 'large-v3-turbo')
        if not fpath:
            return {}, [], {}, render_transcript_ui([], {}, {}, mkey, 0.0)
        
        t0 = time.time()
        data = load_16k_from_file(fpath)
        if len(data) < 4000:
            return {}, [], {}, render_transcript_ui([], {}, {}, mkey, 0.0)

        model = get_model(mkey)
        target_lang, is_urdu_script, is_english_only = LANG_MAP.get(lang_choice, ('ur', False, False))
        is_roman = (lang_choice == 'Roman Urdu (Latin)')
        thresh = float(thresh_pct) / 100.0
        active_prompt = PROMPT_MAP.get(lang_choice, PROMPT_MAP['Bilingual (Urdu + English)'])

        try:
            segs, info = model.transcribe(
                data,
                language=target_lang,
                beam_size=1, best_of=1, temperature=0.0,
                initial_prompt=active_prompt,
                condition_on_previous_text=False,
                without_timestamps=False,
                no_speech_threshold=0.6,
                compression_ratio_threshold=2.4,
                vad_filter=True,
                vad_parameters=dict(min_silence_duration_ms=int(vad_ms) if vad_ms else 300)
            )
        except Exception as e:
            return {}, [], {}, f"<div class='vd-empty'><p>Transcription error: {e}</p></div>", "", "", ""

        segments_list = []
        profiles = {}
        names = {}
        embeddings = {}
        last_sid = None
        last_stime = 0.0
        
        for s in segs:
            raw = s.text.strip()
            if not raw: continue
            conf = float(getattr(s, "avg_logprob", getattr(s, "avg_log_prob", 0.0)))
            no_speech_prob = float(getattr(s, "no_speech_prob", 0.0))
            if no_speech_prob > 0.65 or conf < -1.2: continue
            raw = re.sub(r'\b(\w+(?:\s+\w+){0,3})(?:\s+\1\b)+', r'\1', raw, flags=re.IGNORECASE).strip()
            raw = sanitize_script(raw)
            if not raw or len(raw) < 2: continue

            if is_urdu_script:
                txt_clean = UrduNormalizer.normalize(raw)
            elif is_english_only:
                txt_clean = post_process_english(raw)
            elif is_roman:
                norm_ur = UrduNormalizer.normalize(raw)
                txt_clean = post_process_english(to_roman_urdu(norm_ur))
            else:
                words = raw.split()
                proc = []
                for w in words:
                    if any('\u0600' <= ch <= '\u06FF' for ch in w):
                        proc.append(UrduNormalizer.normalize(w))
                    else:
                        proc.append(w)
                txt_clean = post_process_english(" ".join(proc))

            if not txt_clean or len(txt_clean.strip()) == 0:
                continue

            s0, s1 = s.start, s.end
            seg_dur = s1 - s0
            chunk = data[int(s0*16000):int(s1*16000)]
            spk = 1
            if len(chunk) >= 8000:
                try:
                    with torch.inference_mode():
                        w = torch.from_numpy(chunk).float().unsqueeze(0).to(device_type)
                        e = embedder.encode_batch(w).squeeze().detach().cpu().numpy()
                        en = e / (np.linalg.norm(e) or 1.)
                    
                    spk, score, last_sid, last_stime = identify_speaker(
                        en, seg_dur, s1, embeddings, last_sid, last_stime, threshold=thresh
                    )
                    profiles[spk] = len(embeddings.get(spk, [1]))
                except Exception: pass
            elif last_sid is not None:
                spk = last_sid

            if spk not in names:
                names[spk] = f"Speaker {spk}"

            ts = f"{int(s0//60):02d}:{int(s0%60):02d}"
            segments_list.append({
                "speaker": names[spk],
                "id": spk,
                "start": s0,
                "end": s1,
                "time": ts,
                "text": txt_clean
            })

        elapsed = time.time() - t0
        t_html, s_html, exp_html, plain_txt = render_transcript_ui(segments_list, profiles, names, mkey, elapsed)
        return names, segments_list, profiles, t_html, s_html, exp_html, plain_txt

    transcribe_file_btn.click(
        fn=handle_file_upload,
        inputs=[audio_file, model_dd, lang_dd, thresh_sl, vad_sl],
        outputs=[speaker_names_state, session_segments_state, speaker_profiles_state, transcript_out, sidebar_out, export_html_out, transcript_state]
    )

    ai_btn.click(fn=gemini_summary, inputs=[transcript_state, gemini_key], outputs=[ai_out])

demo.queue(max_size=20).launch(share=True, inline=True, debug=False, show_error=True)
